In [1]:
# tensorflow: Deep learning framework used to build and train the neural network model
import tensorflow as tf
# Sequential: Keras model wrapper that allows building a model layer-by-layer linearly
from tensorflow.keras.models import Sequential
# Dense: Fully connected layer; Input: Defines the shape of the input data
from tensorflow.keras.layers import Dense, Input
# pandas: Library used for data manipulation, loading, and cleaning the CSV dataset
import pandas as pd
# numpy: Library used for numerical operations on arrays
import numpy as np
# train_test_split: Utility to split dataset into training and testing sets
from sklearn.model_selection import train_test_split

In [2]:
# Load the student performance dataset from a CSV file. The columns are separated by semicolons (sep=';').
data = pd.read_csv('student-mat.csv', sep=';')

In [3]:
# Create a binary classification target 'result' (1 for pass, 0 for fail) based on whether final grade G3 >= 10.
data['result'] = (data['G3'] >= 10).astype(int)


In [4]:
# Drop G1, G2, and G3 columns to avoid target leakage since G3 is directly related to the target and G1/G2 are intermediate grades.
data = data.drop(['G1', 'G2', 'G3'], axis=1)

In [5]:
# Convert categorical variables into dummy/indicator variables (one-hot encoding) and drop the first category to avoid multicollinearity.
data = pd.get_dummies(data, drop_first=True)

In [12]:
# Separate features (X) and target variable (y)
X = data.drop('result', axis=1)
y = data['result']

# Split the dataset into 80% training and 20% testing sets using a fixed random state for reproducibility
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
# Convert features to float32 datatype for compatibility with TensorFlow layers
X_train = X_train.astype('float32')
X_test = X_test.astype('float32')

In [7]:
# Define a Sequential neural network model
model = Sequential([
    # Input layer specifying the input shape (number of features in X_train)
    Input(shape=(X_train.shape[1],)),
    # First hidden layer with 64 units and ReLU activation function to introduce non-linearity
    Dense(64, activation='relu'),
    # Second hidden layer with 32 units and ReLU activation function
    Dense(32, activation='relu'),
    # Output layer with 1 unit and Sigmoid activation function to output a probability between 0 and 1 for binary classification
    Dense(1, activation='sigmoid')
])

In [9]:
# Compile the model with the Adam optimizer, binary cross-entropy loss (ideal for binary classification), and track accuracy metric
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)


In [10]:
# Display the architecture summary of the model, showing layer types, output shapes, and parameter counts
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_3 (Dense)                 │ (None, 64)             │         2,560 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,673 (18.25 KB)

 Trainable params: 4,673 (18.25 KB)

 Non-trainable params: 0 (0.00 B)

In [13]:
# Train the model on training data for 20 epochs with a batch size of 16, using 20% of training data for validation
model.fit(
    X_train,
    y_train,
    epochs=20,
    batch_size=16,
    validation_split=0.2
)


Epoch 1/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.6905 - loss: 0.6770 - val_accuracy: 0.6094 - val_loss: 0.7068
Epoch 2/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6984 - loss: 0.6313 - val_accuracy: 0.6406 - val_loss: 0.6729
Epoch 3/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7103 - loss: 0.6102 - val_accuracy: 0.6406 - val_loss: 0.6588
Epoch 4/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7024 - loss: 0.5886 - val_accuracy: 0.6719 - val_loss: 0.6575
Epoch 5/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7302 - loss: 0.5710 - val_accuracy: 0.6562 - val_loss: 0.6570
Epoch 6/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7381 - loss: 0.5527 - val_accuracy: 0.6719 - val_loss: 0.6880
Epoch 7/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7460 - loss: 0.5639 - val_accuracy: 0.6094 - val_loss: 0.6476
Epoch 8/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7500 - loss: 0.5283 - val_accuracy: 0.6094 - val_loss

In [14]:
# Evaluate the trained model's performance on the unseen test dataset
loss, accuracy = model.evaluate(X_test, y_test)
# Print the final testing accuracy
print("Test Accuracy:", accuracy)

3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.6835 - loss: 0.6221 
Test Accuracy: 0.6835442781448364


In [15]:
# Generate predictions for the first 5 test samples
predictions = model.predict(X_test[:5])

# Compare the actual labels with the predicted labels (using a threshold of 0.5 for binary classification)
for i in range(5):
    pred_label = 1 if predictions[i] > 0.5 else 0
    print("Actual:", y_test.values[i], "| Predicted:", pred_label)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step
Actual: 1 | Predicted: 0
Actual: 1 | Predicted: 1
Actual: 0 | Predicted: 1
Actual: 1 | Predicted: 1
Actual: 0 | Predicted: 0
